*0.2 Math / ML basics*

# regularization

**The situation.** Finance wants latency predicted from request features: token counts, time of day, model, region, 60 features in all. You have 80 logged requests. A plain linear fit is perfect on those 80 and useless on the next 80 — with 60 features and 80 examples, the model can explain the noise. More data is months away.

**Regularisation.** Add a penalty to the loss for large parameter values. The model then has to *earn* every big weight with a real pattern; noise is not worth the penalty. In scikit-learn's `Ridge` the strength is `alpha`. In PyTorch it is `weight_decay` on the optimizer. In neural networks, *dropout* — randomly switching off parts of the network during training — has the same effect.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Pick the strength on the validation set.** The data is made so the truth is known: only 5 of the 60 features matter, and the noise is 20 ms. The error should get close to 20 ms and no closer.

In [2]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
features = rng.standard_normal((160, 60))
true_weights = np.zeros(60)
true_weights[:5] = [40, 30, -25, 20, 15]  # only five features really affect latency
latency_ms = (
    300 + features @ true_weights + rng.standard_normal(160) * 20
)  # 20 ms of noise nobody can predict
train_x, val_x, train_y, val_y = train_test_split(
    features, latency_ms, test_size=0.5, random_state=0
)

print(f"{'alpha':>8}{'train error':>14}{'val error':>12}{'largest weight':>16}")
best = None
for alpha in (0.001, 1, 10, 30, 100, 1000):
    model = Ridge(alpha=alpha).fit(train_x, train_y)
    train_error = root_mean_squared_error(train_y, model.predict(train_x))
    val_error = root_mean_squared_error(val_y, model.predict(val_x))
    print(
        f"{alpha:>8}{train_error:>11.1f} ms{val_error:>9.1f} ms{np.abs(model.coef_).max():>16.1f}"
    )
    if best is None or val_error < best[1]:
        best = (alpha, val_error)
print("chosen by validation: alpha =", best[0], f"({best[1]:.1f} ms error, noise floor 20 ms)")
assert best[0] not in (0.001, 1000)

   alpha   train error   val error  largest weight
   0.001       12.6 ms     44.7 ms            45.0
       1       12.8 ms     35.6 ms            42.3
      10       16.6 ms     26.9 ms            33.1
      30       23.3 ms     32.4 ms            24.5
     100       35.1 ms     44.8 ms            14.0
    1000       53.4 ms     61.0 ms             2.4
chosen by validation: alpha = 10 (26.9 ms error, noise floor 20 ms)


**Reading the output.** No penalty: tiny training error, large validation error — it fitted the noise. Huge penalty: both errors large — it cannot fit the signal either. In between, validation error comes close to the 20 ms noise floor. The largest weight shrinks steadily as the penalty grows; the chosen model has weights near the true 40/30/25.

```
error
  │ ●                                    train
  │   ●                              ●
  │     ●         ○             ●
  │       ●   ○       ○     ●
  │         ●   ●  ●    ○ ○              validation: U-shape, pick the bottom
  └──────────────────────────────────── stronger penalty (alpha) →
```

**The same knob in PyTorch.** One argument — this is what fine-tuning scripts set.

In [3]:
import torch

layer = torch.nn.Linear(1536, 3)
optimizer = torch.optim.AdamW(
    layer.parameters(), lr=1e-3, weight_decay=0.01
)  # weight_decay = the penalty
dropout = torch.nn.Dropout(p=0.1)  # switches off 10% of values during training
print("AdamW weight_decay:", optimizer.param_groups[0]["weight_decay"], "| dropout p:", dropout.p)
assert optimizer.param_groups[0]["weight_decay"] == 0.01

AdamW weight_decay: 0.01 | dropout p: 0.1


**The rule to remember.** Regularisation trades a little training accuracy for better validation accuracy. Choose its strength on the validation set — never on the training set, where less penalty always looks better.

| Use it when | Don't when | Instead use |
|---|---|---|
| overfitting with a training/validation gap; many features, few examples | underfitting (both errors high) — more penalty makes it worse | a bigger model or more features |

**Watch out**
- More data is the strongest regulariser there is; if you can get it, get it.
- Dropout is on in training and off at inference; `model.eval()` handles that — forgetting it makes predictions noisy.
- LoRA fine-tuning (later in this repo) regularises by limiting how many parameters can move at all.